In [11]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_path = Path("SystemDesignInterview.pdf")
if not pdf_path.exists():
    pdf_path = Path.cwd() / "SystemDesignInterview.pdf"

if not pdf_path.exists():
    raise FileNotFoundError(
        f"PDF file was not found. Expected it in the notebook folder: {pdf_path}"
    )

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

if not documents:
    raise ValueError(f"No pages were loaded from the PDF: {pdf_path}")

print(f"PDF path: {pdf_path}")
print("Number of pages:", len(documents))
print(documents[0].page_content[:1000])

PDF path: SystemDesignInterview.pdf
Number of pages: 269



In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
)
chunks = splitter.split_documents(documents)

print("Documents:", len(documents))
print("Chunks:", len(chunks))

Documents: 269
Chunks: 426


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)
print("Local embedding model initialized successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4891.51it/s]


Local embedding model initialized successfully.


In [14]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
)

print("Chroma vector store created successfully.")

Chroma vector store created successfully.


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [ ]:
query = "What are the three important characteristics of a large system?"

results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)
    print("-----")

Reference materials
[1] Erlang at Facebook: 
https://www.erlang-
factory.com/upload/presentations/31/EugeneLetuchy-ErlangatFacebook.pdf
[2] Messenger and WhatsApp process 60 billion messages a day:
https://www.theverge.com/2016/4/12/11415198/facebook-messenger-whatsapp-number-
messages-vs-sms-f8-2016
[3] Long tail: 
https://en.wikipedia.org/wiki/Long_tail
[4] The Underlying Technology of Messages: 
https://www.facebook.com/notes/facebook-
engineering/the-underlying-technology-of-messages/454991608919/
[5] How Discord Stores Billions of Messages: 
https://blog.discordapp.com/how-discord-
stores-billions-of-messages-7fa6ec7ee4c7
[6] Announcing Snowflake: 
https://blog.twitter.com/engineering/en_us/a/2010/announcing-
snowflake.html
[7] Apache ZooKeeper: 
https://zookeeper.apache.org/
[8] From nothing: the evolution of WeChat background system (Article in Chinese):
https://www.infoq.cn/article/the-road-of-the-growth-weixin-background
[9] End-to-end encryption:
-----
Reference materials
[1]

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    temperature=0
)

In [20]:
question = "What are the three important characteristics of a large system?"

docs = retriever.invoke(question)

print("NUMBER OF RETRIEVED DOCUMENTS:", len(docs))

for i, doc in enumerate(docs):
    print(f"\n========== CHUNK {i+1} ==========")
    print("SOURCE:", doc.metadata)
    print(doc.page_content[:2000])

NUMBER OF RETRIEVED DOCUMENTS: 3

========== CHUNK 1 ==========
SOURCE: {'page_label': '130', 'author': 'Alex Xu', 'page': 129, 'total_pages': 269, 'producer': 'calibre 3.9.0 [https://calibre-ebook.com]', 'title': "System Design Interview – An insider's guide, Second Edition: Step by Step Guide, Tips and 15 System Design Interview Questions with Detailed Solutions", 'source': 'SystemDesignInterview.pdf', 'creator': 'calibre 3.9.0 [https://calibre-ebook.com]', 'creationdate': '2020-10-16T23:12:01+00:00'}
Availability, consistency, and reliability. These concepts are at the core of any large
system’s success. We discussed them in detail in Chapter 1, please refresh your memory
on these topics.
Congratulations on getting this far! Now give yourself a pat on the back. Good job!

========== CHUNK 2 ==========
SOURCE: {'author': 'Alex Xu', 'title': "System Design Interview – An insider's guide, Second Edition: Step by Step Guide, Tips and 15 System Design Interview Questions with Detailed So

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    temperature=0
)

context = "\n\n".join(
    doc.page_content for doc in docs
)

prompt = f"""
You are answering questions about a book.

Answer the question using ONLY the provided context.

Context:
{context}

Question:
{question}

If the answer cannot be found in the context, say:
"I don't know based on the provided document."
"""

response = llm.invoke(prompt)

print(response.content)